# E5 cosine and sentence coverage on Colab GPU (Samer's extra metrics)

Runs `scripts/score_neural_extra.py` for the full Ship of Theseus analysis. It computes the **E5 document cosine** (`intfloat/e5-base-v2`, "query: " prefix, 512-token sentence-packed chunks, token-weighted) and **MPNet sentence coverage** (share of original sentences whose best match in the rewrite reaches cosine 0.6). The CPU metrics (chrF, word edit rate, exact equality, entity/number/quantity recall) run locally.

**Before running**
1. `MyDrive/sot_colab/sot_colab_bundle.zip` is already on Drive from the semantic run; it provides the corpus.
2. On the Mac, after `scripts/info_features.py` has finished, build the small extra bundle:
   ```bash
   zip -r -9 data/colab/sot_extra_bundle.zip src scripts data/derived/sot_sentences.parquet
   ```
   Upload it to **`MyDrive/sot_colab/`**.
3. **Runtime → Change runtime type → A100 GPU**, then run all cells. Run it in fp32, like the semantic scores.

**Outputs** go to `MyDrive/sot_colab/derived/`: `sot_pairs_neural_extra.parquet` and `sot_floors_neural_extra.parquet`. Download both into the repo's `data/derived/`.

**Disconnects:** run all cells again; it resumes from the Drive caches.

In [ ]:
# 1. Drive, GPU, unpack (corpus from the semantic bundle, code + sentences from the extra bundle)
import os, shutil, subprocess, sys, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/sot_colab")
DERIVED, WORK = DRIVE / "derived", Path("/content/work")
for z in ("sot_colab_bundle.zip", "sot_extra_bundle.zip"):
    assert (DRIVE / z).exists(), f"upload {z} to {DRIVE} first"

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> A100 GPU"
GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(0)} ({GPU_MEM:.0f} GB)")

if not (WORK / "data/sot/paraphrased_datasets").exists():
    shutil.unpack_archive(DRIVE / "sot_colab_bundle.zip", WORK)
shutil.unpack_archive(DRIVE / "sot_extra_bundle.zip", WORK)   # always: newest code overrides the old bundle's
DERIVED.mkdir(parents=True, exist_ok=True)
for name in ("sot_sample.parquet", "sot_floors.parquet", "sot_sentences.parquet"):
    if not (DERIVED / name).exists() or name == "sot_sentences.parquet":
        shutil.copy2(WORK / "data/derived" / name, DERIVED / name)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score==0.3.13"], check=True)

BATCH = {"jobs_per_batch": 32, "batch_size": 64} if GPU_MEM > 30 else \
        {"jobs_per_batch": 16, "batch_size": 32} if GPU_MEM > 20 else {"jobs_per_batch": 8, "batch_size": 16}
print("ready; batch settings:", BATCH)

In [ ]:
# 2. Full run (resumable, fp32)
import pandas as pd

cmd = [sys.executable, "scripts/score_neural_extra.py", "--derived", str(DERIVED), "--dtype", "fp32",
       "--jobs-per-batch", str(BATCH["jobs_per_batch"]), "--batch-size", str(BATCH["batch_size"])]
env = {**os.environ, "PYTHONPATH": str(WORK / "src"), "TOKENIZERS_PARALLELISM": "false"}
t = time.time()
with open(DERIVED / "neural_extra.log", "a") as f, subprocess.Popen(
        cmd, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        if any(k in line for k in ("jobs,", "done", "originals", "Error", "Traceback", "error")):
            print(line, end="", flush=True)
        f.write(line)
if p.returncode:
    raise RuntimeError(f"score_neural_extra.py failed (exit {p.returncode}); see {DERIVED / 'neural_extra.log'}")

pairs_x = pd.read_parquet(DERIVED / "sot_pairs_neural_extra.parquet")
floors_x = pd.read_parquet(DERIVED / "sot_floors_neural_extra.parquet")
sample = pd.read_parquet(DERIVED / "sot_sample.parquet")
n_orig = pairs_x[["dataset", "source", "key"]].drop_duplicates().shape[0]
print(f"{(time.time() - t) / 60:.0f} min | pairs: {len(pairs_x):,} rows for {n_orig:,} / {len(sample):,} originals | "
      f"floors: {len(floors_x):,} / 1,400")
assert n_orig == len(sample) and len(floors_x) == 1400, "incomplete: run this cell again to resume"
print(f"Done. Download sot_pairs_neural_extra.parquet and sot_floors_neural_extra.parquet from {DERIVED} "
      "into the repo's data/derived/.")